# Session 6 - Frozen Sub-Circuit Experiment

Previous experiments demonstrated that shared initialization significantly improves quantum parameter aggregation.

This experiment investigates whether explicitly sharing a frozen quantum sub-circuit across all clients can further improve aggregation.

Most circuit parameters remain fixed and shared, while only a small subset of parameters is trained locally.

This setup serves as a simplified quantum analogue of the shared-structure principle used in RAVAN.

In [2]:
import numpy as np

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.utils.loss_functions import CrossEntropyLoss

from qiskit_algorithms.optimizers import COBYLA

In [3]:
# --------------------------------------------------
# Dataset
# --------------------------------------------------

X, y = make_moons(
    n_samples=400,
    noise=0.15,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# --------------------------------------------------
# Non-IID Federated Split
# --------------------------------------------------

client1_idx = np.where(y_train == 0)[0][:128]
client1_idx = np.concatenate([
    client1_idx,
    np.where(y_train == 1)[0][:32]
])

client2_idx = np.where(y_train == 0)[0][128:158]
client2_idx = np.concatenate([
    client2_idx,
    np.where(y_train == 1)[0][32:160]
])

X_client1 = X_train[client1_idx]
y_client1 = y_train[client1_idx]

X_client2 = X_train[client2_idx]
y_client2 = y_train[client2_idx]

# Convert labels to {-1,+1}

y_client1_q = 2 * y_client1 - 1
y_client2_q = 2 * y_client2 - 1
y_test_q = 2 * y_test - 1

print("Client 1 samples:", len(X_client1))
print("Client 2 samples:", len(X_client2))
print("Test samples:", len(X_test))

Client 1 samples: 160
Client 2 samples: 158
Test samples: 80


In [4]:
num_qubits = 4

x_params = ParameterVector("x", 2)

# Shared frozen parameters
frozen_theta = ParameterVector("f", 6)

# Trainable adapter parameters
adapter_theta = ParameterVector("a", 2)

qc = QuantumCircuit(num_qubits)

# Data Encoding
qc.ry(x_params[0], 0)
qc.ry(x_params[1], 1)

# Frozen shared backbone
for i in range(3):
    qc.ry(frozen_theta[2*i], i)
    qc.rz(frozen_theta[2*i + 1], i)

# Trainable adapter on last qubit
qc.ry(adapter_theta[0], 3)
qc.rz(adapter_theta[1], 3)

# Entanglement
qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)

observable = SparsePauliOp.from_list([
    ("ZZZZ", 1)
])

print(qc)

     ┌──────────┐┌──────────┐┌──────────┐               
q_0: ┤ Ry(x[0]) ├┤ Ry(f[0]) ├┤ Rz(f[1]) ├──■────────────
     ├──────────┤├──────────┤├──────────┤┌─┴─┐          
q_1: ┤ Ry(x[1]) ├┤ Ry(f[2]) ├┤ Rz(f[3]) ├┤ X ├──■───────
     ├──────────┤├──────────┤└──────────┘└───┘┌─┴─┐     
q_2: ┤ Ry(f[4]) ├┤ Rz(f[5]) ├─────────────────┤ X ├──■──
     ├──────────┤├──────────┤                 └───┘┌─┴─┐
q_3: ┤ Ry(a[0]) ├┤ Rz(a[1]) ├──────────────────────┤ X ├
     └──────────┘└──────────┘                      └───┘


In [5]:
shared_frozen_weights = np.random.rand(6)

print("Shared Frozen Parameters:")
print(shared_frozen_weights)

Shared Frozen Parameters:
[0.48625202 0.50966299 0.74521083 0.65616257 0.51834433 0.21377528]


In [7]:
frozen_dict = {
    frozen_theta[i]: shared_frozen_weights[i]
    for i in range(6)
}

qc_frozen = qc.assign_parameters(
    frozen_dict
)

print(
    "Remaining parameters:",
    len(qc_frozen.parameters)
)

print(qc_frozen.parameters)

Remaining parameters: 4
ParameterView([ParameterVectorElement(a[0]), ParameterVectorElement(a[1]), ParameterVectorElement(x[0]), ParameterVectorElement(x[1])])


In [8]:
qnn = EstimatorQNN(
    circuit=qc_frozen,
    input_params=x_params,
    weight_params=adapter_theta,
    observables=observable
)

print("Frozen-Subcircuit QNN Ready")

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Frozen-Subcircuit QNN Ready


In [9]:
shared_adapter_init = np.random.rand(2)

print("Shared Adapter Initialization:")
print(shared_adapter_init)

Shared Adapter Initialization:
[0.58312459 0.81474069]


In [10]:
classifier_client1 = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=COBYLA(maxiter=100),
    loss=CrossEntropyLoss(),
    one_hot=False,
    initial_point=shared_adapter_init
)

classifier_client1.fit(
    X_client1,
    y_client1_q
)

print("Client 1 training complete!")

Client 1 training complete!


In [11]:
classifier_client2 = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=COBYLA(maxiter=100),
    loss=CrossEntropyLoss(),
    one_hot=False,
    initial_point=shared_adapter_init
)

classifier_client2.fit(
    X_client2,
    y_client2_q
)

print("Client 2 training complete!")

Client 2 training complete!


In [12]:
print("Client 1 Adapter Weights:")
print(classifier_client1.weights)

print("\nClient 2 Adapter Weights:")
print(classifier_client2.weights)

Client 1 Adapter Weights:
[1.57080539 0.81477912]

Client 2 Adapter Weights:
[-4.93054717e-05  2.40021884e+00]


In [13]:
adapter_global_weights = (
    classifier_client1.weights +
    classifier_client2.weights
) / 2

print("FedAvg Adapter Weights:")
print(adapter_global_weights)

FedAvg Adapter Weights:
[0.78537804 1.60749898]


In [14]:
predictions = []

for sample in X_test:

    output = qnn.forward(
        input_data=sample,
        weights=adapter_global_weights
    )

    pred = 1 if output[0][0] >= 0 else -1

    predictions.append(pred)

predictions = np.array(predictions)

frozen_accuracy = np.mean(
    predictions == y_test_q
)

print("Frozen Sub-Circuit FedAvg Accuracy:")
print(frozen_accuracy)

Frozen Sub-Circuit FedAvg Accuracy:
0.675
